


**Workflow :**
1. Choisir une image STMD → GT chargé automatiquement
2. Extraire les features (4 stages + attention globale)
3. Visualiser **PCA RGB** par stage (ToggleButtons)
4. Explorer les **cartes d'attention** des blocs globaux (sliders px/py/tête)
5. Lancer le **t-SNE** par stage
6. Inspecter la **variance expliquée** PCA

In [1]:
import os, sys, time, warnings, zipfile, tempfile
import numpy as np
import torch
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from PIL import Image
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import ipywidgets as widgets
from IPython.display import display, clear_output
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white'})

ROOT     = Path('..').resolve()
SAM2_DIR = ROOT / 'TextureSAM' / 'sam2'
if str(SAM2_DIR) not in sys.path:
    sys.path.insert(0, str(SAM2_DIR))

IMG_DIR  = ROOT / 'data' / 'raw' / 'stmd' / 'images'
LBL_DIR  = ROOT / 'data' / 'raw' / 'stmd' / 'labels'

SEED        = 42
IMAGE_SIZE  = 1024
_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
_STD  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

GLOBAL_ATT_BLOCKS = (7, 10, 13)   # Hiera Small — tous dans Stage 3
STAGE_NAMES = ['Stage 1', 'Stage 2', 'Stage 3', 'Stage 4']
CONV_IDX    = {'Stage 1': 3, 'Stage 2': 2, 'Stage 3': 1, 'Stage 4': 0}
STAGE_SIZES = {'Stage 1': 256, 'Stage 2': 128, 'Stage 3': 64, 'Stage 4': 32}
PALETTE = ['#e6194b','#3cb44b','#4363d8','#f58231','#911eb4',
           '#42d4f4','#f032e6','#bfef45','#fabed4','#469990']

np.random.seed(SEED)
torch.manual_seed(SEED)
print('Imports OK')

Imports OK


## ⚙️ Chargement du modèle

In [2]:
def _build_encoder():
    from sam2.modeling.backbones.hieradet import Hiera
    from sam2.modeling.backbones.image_encoder import ImageEncoder, FpnNeck
    from sam2.modeling.position_encoding import PositionEmbeddingSine
    trunk = Hiera(embed_dim=96, num_heads=1, stages=(1,2,11,2),
                  global_att_blocks=(7,10,13),
                  window_pos_embed_bkg_spatial_size=(7,7))
    neck = FpnNeck(
        position_encoding=PositionEmbeddingSine(
            num_pos_feats=256, normalize=True, scale=None, temperature=10000),
        d_model=256, backbone_channel_list=[768,384,192,96],
        kernel_size=1, stride=1, padding=0,
        fpn_interp_model='nearest', fuse_type='sum', fpn_top_down_levels=[2,3])
    return ImageEncoder(trunk=trunk, neck=neck, scalp=1)

def _load_ckpt():
    ckpt_pt  = ROOT / 'checkpoints' / 'sam2.1_hiera_small_1.pt'
    ckpt_dir = ROOT / 'checkpoints' / 'sam2.1_hiera_small_1'
    if ckpt_pt.is_file():
        try:
            sd = torch.load(ckpt_pt, map_location='cpu', weights_only=True)
            return sd.get('model', sd)
        except Exception:
            pass
    archive = ckpt_dir / 'archive' if (ckpt_dir / 'archive').is_dir() else ckpt_dir
    if archive.is_dir():
        with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as tmp:
            tmp_path = tmp.name
        with zipfile.ZipFile(tmp_path, 'w', zipfile.ZIP_STORED) as zf:
            for fp in archive.rglob('*'):
                if fp.is_file():
                    info = zipfile.ZipInfo(str(fp.relative_to(archive.parent)))
                    info.date_time = (1980,1,1,0,0,0)
                    zf.writestr(info, fp.read_bytes())
        sd = torch.load(tmp_path, map_location='cpu', weights_only=False)
        os.unlink(tmp_path)
        return sd.get('model', sd)
    return None

device  = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = _build_encoder()
sd      = _load_ckpt()
if sd:
    prefix = 'image_encoder.'
    if any(k.startswith(prefix) for k in sd):
        sd = {k[len(prefix):]: v for k,v in sd.items() if k.startswith(prefix)}
    miss, unex = encoder.load_state_dict(sd, strict=False)
    print(f'Checkpoint charge ({len(miss)} manquantes, {len(unex)} inattendues)')
else:
    print('Poids aleatoires')
encoder = encoder.to(device).eval()
print(f'Encoder sur {device}')

Checkpoint charge (0 manquantes, 0 inattendues)
Encoder sur cuda


In [3]:
# ── Etat global ──────────────────────────────────────────────────────────
STATE = {
    'img_path' : None,
    'lbl_arr'  : None,
    'img_pil'  : None,
    'features' : {},
    'gt_maps'  : {},
    'attn_cache'    : {},
    'features_fused': {},
}

def l2_norm(X):
    norms = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.maximum(norms, 1e-8)

def class_colors(classes):
    return {int(c): PALETTE[i % len(PALETTE)] for i, c in enumerate(classes)}

def preprocess(img_pil):
    img = img_pil.convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE), Image.BILINEAR)
    x   = torch.from_numpy(np.array(img)).float() / 255.0
    x   = x.permute(2, 0, 1)
    x   = (x - _MEAN) / _STD
    return x.unsqueeze(0).to(device)

def stratified_subsample(X, y, max_pts):
    rng = np.random.default_rng(SEED)
    classes, counts = np.unique(y, return_counts=True)
    idx = []
    for cls, cnt in zip(classes, counts):
        cidx  = np.where(y == cls)[0]
        quota = max(1, min(int(round(max_pts * cnt / len(y))), len(cidx)))
        idx.append(rng.choice(cidx, size=quota, replace=False))
    return np.concatenate(idx)

print('Helpers OK')

Helpers OK


---
## 1️⃣  Choisir une image

In [4]:
exts = {'.jpg', '.jpeg', '.png'}
image_files = sorted([p.name for p in IMG_DIR.iterdir()
                       if p.suffix.lower() in exts])

img_dropdown = widgets.Dropdown(
    options=image_files, description='Image :',
    style={'description_width': 'initial'}, layout=widgets.Layout(width='300px'))

load_btn = widgets.Button(description='Charger image',
                           button_style='primary', icon='folder-open')
out_img  = widgets.Output()

def on_load(b):
    name     = img_dropdown.value
    img_path = IMG_DIR / name
    lbl_path = LBL_DIR / (img_path.stem + '.png')
    img_pil  = Image.open(img_path)
    lbl_arr  = np.array(Image.open(lbl_path))
    STATE.update(img_path=img_path, img_pil=img_pil, lbl_arr=lbl_arr,
                 features={}, gt_maps={}, attn_cache={})
    classes, counts = np.unique(lbl_arr, return_counts=True)
    colors = class_colors(classes)
    with out_img:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        axes[0].imshow(np.array(img_pil.convert('RGB')), cmap='gray')
        axes[0].set_title(f'{name}  |  {img_pil.size}  mode={img_pil.mode}',
                           fontsize=10)
        axes[0].axis('off')
        cmap_gt = plt.colormaps['tab10']
        gt_rgb = np.zeros((*lbl_arr.shape, 3))
        for k, cls in enumerate(classes):
            gt_rgb[lbl_arr == cls] = mcolors.to_rgb(cmap_gt(k))
        axes[1].imshow(gt_rgb)
        patches = [plt.Rectangle((0,0),1,1, color=cmap_gt(k),
                    label=f'cls {int(c)} — {int(cnt)} px ({100*cnt/lbl_arr.size:.1f}%)')
                   for k,(c,cnt) in enumerate(zip(classes, counts))]
        axes[1].legend(handles=patches, fontsize=8, loc='lower right')
        axes[1].set_title(f'GT — {len(classes)} classes', fontsize=10)
        axes[1].axis('off')
        plt.suptitle('Image + Ground Truth', fontsize=12, y=1.01)
        plt.tight_layout()
        plt.show()
        print(f'Image chargee. Lance "Extraire features" maintenant.')

load_btn.on_click(on_load)
display(widgets.HBox([img_dropdown, load_btn]), out_img)

Output()

---
## 2️⃣  Extraction des features

In [5]:
extract_btn = widgets.Button(description='Extraire features',
                              button_style='success', icon='bolt')
prog        = widgets.IntProgress(min=0, max=4, bar_style='info',
                                  description='', layout=widgets.Layout(width='300px'))
out_ext     = widgets.Output()

def on_extract(b):
    if STATE['img_pil'] is None:
        with out_ext:
            print('Charger une image d abord.')
        return
    prog.value = 0
    feat_cache = {}
    attn_cache  = {}
    fused_cache = {}
    hooks = []
    # --- hooks FPN neck (4 stages) ---
    for sn, ci in CONV_IDX.items():
        def _fhook(m, inp, out, _n=sn):
            feat_cache[_n] = out.detach().cpu()
        hooks.append(encoder.neck.convs[ci].register_forward_hook(_fhook))
    # --- hooks QK blocs global attention ---
    from sam2.modeling.backbones.hieradet import do_pool
    for bi in GLOBAL_ATT_BLOCKS:
        attn_mod = encoder.trunk.blocks[bi].attn
        def _ahook(m, inp, out, _bi=bi):
            x_in = inp[0]
            B, H, W, C = x_in.shape
            with torch.no_grad():
                qkv = m.qkv(x_in).reshape(B, H*W, 3, m.num_heads, -1)
                q, k, _ = torch.unbind(qkv, 2)
                if m.q_pool:
                    q  = do_pool(q.reshape(B, H, W, -1), m.q_pool)
                    H2, W2 = q.shape[1], q.shape[2]
                    q  = q.reshape(B, H2*W2, m.num_heads, -1)
                else:
                    H2, W2 = H, W
            attn_cache[_bi] = {'q': q[0].detach().cpu(),
                               'k': k[0].detach().cpu(),
                               'H': int(H2), 'W': int(W2)}
        hooks.append(attn_mod.register_forward_hook(_ahook))
    # --- hook FPN neck (features fusionnees apres top-down) ---
    def _nfhook(m, inp, out):
        out_list, _ = out
        # out_list[0]=Stage1(256x256) out_list[1]=Stage2(128x128)
        # out_list[2]=Stage3(64x64) <- seul niveau avec fusion reelle
        fused_cache['Stage 1 fus'] = out_list[0].detach().cpu()
        fused_cache['Stage 2 fus'] = out_list[1].detach().cpu()
        fused_cache['Stage 3 fus'] = out_list[2].detach().cpu()
    hooks.append(encoder.neck.register_forward_hook(_nfhook))
    # --- forward ---
    t0 = time.time()
    with torch.no_grad():
        encoder(preprocess(STATE['img_pil']))
    for h in hooks:
        h.remove()
    # --- post-process ---
    for sn, sz in STAGE_SIZES.items():
        STATE['features'][sn] = feat_cache[sn][0].permute(1,2,0).numpy()
        lbl_r = np.array(Image.fromarray(STATE['lbl_arr'])
                         .resize((sz, sz), Image.NEAREST))
        STATE['gt_maps'][sn] = lbl_r
        prog.value += 1
    FUSED_SIZES = {'Stage 1 fus': 256, 'Stage 2 fus': 128, 'Stage 3 fus': 64}
    for fn, sz in FUSED_SIZES.items():
        t = fused_cache[fn][0].permute(1, 2, 0).numpy()
        STATE['features_fused'][fn] = t
    STATE['attn_cache'] = attn_cache
    with out_ext:
        clear_output(wait=True)
        print(f'Features extraites en {time.time()-t0:.1f}s')
        for sn in STAGE_NAMES:
            print(f'  {sn}: {STATE["features"][sn].shape}')
        print(f'  Attention capturee sur blocs: {list(attn_cache.keys())}')
        print('  Features fusionnees :')
        for fn in ['Stage 1 fus', 'Stage 2 fus', 'Stage 3 fus']:
            print(f'    {fn}: {STATE["features_fused"][fn].shape}')

extract_btn.on_click(on_extract)
display(widgets.HBox([extract_btn, prog]), out_ext)

Output()

---
## 3️⃣  PCA RGB par stage

Sélectionne un stage ou *Tous* pour la vue panoramique.  
Stage 3 ★ est le stage sélectionné pour la segmentation.

In [ ]:
def compute_pca_rgb(feat):
    H, W, D = feat.shape
    X   = l2_norm(feat.reshape(-1, D))
    pca = PCA(n_components=3, random_state=SEED)
    rgb = pca.fit_transform(X)
    for c in range(3):
        mn, mx = rgb[:,c].min(), rgb[:,c].max()
        rgb[:,c] = (rgb[:,c] - mn) / max(mx - mn, 1e-8)
    return rgb.reshape(H, W, 3), pca.explained_variance_ratio_

stage_toggle = widgets.ToggleButtons(
    options=STAGE_NAMES + ['Tous'],
    description='Stage :', button_style='',
    style={'button_width': '100px', 'description_width': 'initial'})
out_pca = widgets.Output()

def show_pca(change=None):
    if not STATE['features']:
        with out_pca:
            print('Extraire les features d abord.')
        return
    sel = stage_toggle.value
    with out_pca:
        clear_output(wait=True)
        if sel == 'Tous':
            fig, axes = plt.subplots(1, 5, figsize=(23, 4.5))
            axes[0].imshow(np.array(STATE['img_pil'].convert('RGB')))
            axes[0].set_title('Original', fontsize=11)
            axes[0].axis('off')
            for i, sn in enumerate(STAGE_NAMES):
                rgb, evr = compute_pca_rgb(STATE['features'][sn])
                sz   = STAGE_SIZES[sn]
                star = ' *' if sn == 'Stage 3' else ''
                fw   = 'bold' if star else 'normal'
                axes[i+1].imshow(rgb)
                axes[i+1].set_title(f'{sn}{star} ({sz}x{sz})\nEV: {evr.sum()*100:.1f}%',
                                     fontsize=9, fontweight=fw)
                axes[i+1].axis('off')
            plt.suptitle('PCA RGB — 4 stages TextureSAM', fontsize=13)
        else:
            fig, axes = plt.subplots(1, 3, figsize=(17, 5))
            feat = STATE['features'][sel]
            rgb, evr = compute_pca_rgb(feat)
            sz  = STAGE_SIZES[sel]
            gt  = STATE['gt_maps'][sel]
            # col 0 : original
            axes[0].imshow(np.array(STATE['img_pil'].convert('RGB')))
            axes[0].set_title('Original', fontsize=11)
            axes[0].axis('off')
            # col 1 : PCA RGB
            axes[1].imshow(rgb)
            axes[1].set_title(
                f'PCA RGB {sel} ({sz}x{sz})\n'
                f'PC1={evr[0]*100:.1f}%  PC2={evr[1]*100:.1f}%  PC3={evr[2]*100:.1f}%',
                fontsize=10)
            axes[1].axis('off')
            # col 2 : GT a la resolution du stage
            cmap_gt = plt.colormaps['tab10']
            classes = np.unique(gt)
            gt_rgb = np.zeros((*gt.shape, 3))
            for k, cls in enumerate(classes):
                gt_rgb[gt == cls] = mcolors.to_rgb(cmap_gt(k))
            axes[2].imshow(gt_rgb)
            axes[2].set_title(f'GT @ {sz}x{sz}', fontsize=10)
            axes[2].axis('off')
            plt.suptitle(f'PCA RGB — {sel}', fontsize=13)
        plt.tight_layout()
        plt.show()

stage_toggle.observe(show_pca, names='value')
display(stage_toggle, out_pca)
show_pca()

ToggleButtons(description='Stage :', options=('Stage 1', 'Stage 2', 'Stage 3', 'Stage 4', 'Tous'), style=Toggl…

Output()

---
## 4️⃣  Cartes d'attention globale

Les 3 blocs global attention de Hiera Small (**7, 10, 13**) sont tous dans **Stage 3** (64×64, 4 têtes).  
Déplace les sliders **x / y** pour changer le pixel query — la carte montre à quoi il fait attention.
Coche *Toutes les têtes* pour voir la moyenne sur les 4 têtes.

In [ ]:
def compute_attn_for_pixel(block_idx, head_idx, px, py):
    cache = STATE['attn_cache'].get(block_idx)
    if cache is None:
        return None
    q  = cache['q']         # (HW, nheads, head_dim)
    k  = cache['k']
    H, W = cache['H'], cache['W']
    q_idx = py * W + px
    q_vec = q[q_idx, head_idx, :]     # (head_dim,)
    k_all = k[:, head_idx, :]          # (HW, head_dim)
    scale  = q_vec.shape[-1] ** -0.5
    scores = torch.einsum('d,nd->n', q_vec, k_all) * scale
    return torch.softmax(scores, dim=0).numpy().reshape(H, W)

block_sel = widgets.Dropdown(
    options=[('Bloc 7  (global att 1)', 7),
             ('Bloc 10 (global att 2)', 10),
             ('Bloc 13 (global att 3)', 13)],
    description='Bloc :', style={'description_width':'initial'},
    layout=widgets.Layout(width='260px'))

head_sel = widgets.Dropdown(
    options=[(f'Tete {i}', i) for i in range(4)],
    description='Tete :', style={'description_width':'initial'},
    layout=widgets.Layout(width='160px'))

px_slider = widgets.IntSlider(min=0, max=63, value=32,
    description='x (col) :', continuous_update=False,
    style={'description_width':'initial'}, layout=widgets.Layout(width='350px'))

py_slider = widgets.IntSlider(min=0, max=63, value=32,
    description='y (row) :', continuous_update=False,
    style={'description_width':'initial'}, layout=widgets.Layout(width='350px'))

avg_cb = widgets.Checkbox(value=False, description='Toutes les tetes (moyenne)')
out_attn = widgets.Output()

def show_attn(change=None):
    if not STATE['attn_cache']:
        with out_attn:
            print('Extraire les features d abord.')
        return
    blk = block_sel.value
    hd  = head_sel.value
    px  = px_slider.value
    py  = py_slider.value
    avg = avg_cb.value
    cache = STATE['attn_cache'].get(blk)
    if cache is None:
        return
    H, W   = cache['H'], cache['W']
    nheads = cache['q'].shape[1]
    if avg:
        maps = [compute_attn_for_pixel(blk, h, px, py) for h in range(nheads)]
        attn_map = np.mean([m for m in maps if m is not None], axis=0)
    else:
        attn_map = compute_attn_for_pixel(blk, hd, px, py)
    if attn_map is None:
        return
    # Coordonnees originales
    orig_h, orig_w = STATE['lbl_arr'].shape
    px0 = int(px * orig_w / W)
    py0 = int(py * orig_h / H)
    img_rgb = np.array(STATE['img_pil'].convert('RGB'))
    # Upscale attention vers image originale
    attn_up = np.array(Image.fromarray(
        (attn_map / attn_map.max() * 255).astype(np.uint8)
    ).resize((orig_w, orig_h), Image.BILINEAR)) / 255.0
    with out_attn:
        clear_output(wait=True)
        fig = plt.figure(figsize=(21, 5))
        ax1 = fig.add_subplot(1, 4, 1)
        ax2 = fig.add_subplot(1, 4, 2)
        ax3 = fig.add_subplot(1, 4, 3)
        ax4 = fig.add_subplot(1, 4, 4)
        # --- Image avec point query ---
        ax1.imshow(img_rgb)
        ax1.plot(px0, py0, 'r+', markersize=16, markeredgewidth=2.5)
        ax1.set_title('Image (+ = pixel query)', fontsize=10)
        ax1.axis('off')
        # --- Carte d attention brute ---
        im = ax2.imshow(attn_map, cmap='inferno', interpolation='nearest')
        ax2.plot(px, py, 'b+', markersize=10, markeredgewidth=2)
        head_lbl = 'moy. tetes' if avg else f'Tete {hd}'
        ax2.set_title(f'Attention Bloc {blk} ({head_lbl})\n64x64', fontsize=10)
        ax2.axis('off')
        plt.colorbar(im, ax=ax2, fraction=0.046, pad=0.04)
        # --- Overlay sur image ---
        ax3.imshow(img_rgb)
        ax3.imshow(attn_up, cmap='inferno', alpha=0.55)
        ax3.plot(px0, py0, 'w+', markersize=14, markeredgewidth=2.5)
        ax3.set_title('Attention superposee sur image', fontsize=10)
        ax3.axis('off')
        # --- Miniatures 4 tetes ---
        ax4.axis('off')
        ax4.set_title(f'4 tetes independantes', fontsize=10)
        positions = [(0.01, 0.51, 0.47, 0.47),
                     (0.52, 0.51, 0.47, 0.47),
                     (0.01, 0.01, 0.47, 0.47),
                     (0.52, 0.01, 0.47, 0.47)]
        for h_i in range(min(nheads, 4)):
            m = compute_attn_for_pixel(blk, h_i, px, py)
            if m is not None:
                sub = ax4.inset_axes(positions[h_i])
                sub.imshow(m, cmap='inferno', interpolation='nearest')
                border = 'red' if (not avg and h_i == hd) else 'white'
                for sp in sub.spines.values():
                    sp.set_edgecolor(border); sp.set_linewidth(2)
                sub.set_title(f'H{h_i}', fontsize=7)
                sub.set_xticks([]); sub.set_yticks([])
        plt.suptitle(
            f'Attention globale — Bloc {blk} — query ({px}, {py})',
            fontsize=12)
        plt.tight_layout()
        plt.show()

for w in [block_sel, head_sel, px_slider, py_slider, avg_cb]:
    w.observe(show_attn, names='value')

display(
    widgets.HBox([block_sel, head_sel, avg_cb]),
    widgets.HBox([px_slider, py_slider]),
    out_attn)
show_attn()

Output()

---
## 5️⃣  t-SNE colorié par classe GT

Sous-échantillonnage stratifié (max 2000 pts/stage) → PCA 50d → t-SNE 2d.  
Sélectionne un stage ou *Tous* puis clique **Lancer t-SNE**.

In [ ]:
def run_tsne(stage_name, max_pts=2000):
    feat = STATE['features'][stage_name]
    gt   = STATE['gt_maps'][stage_name]
    X    = l2_norm(feat.reshape(-1, feat.shape[-1]))
    y    = gt.flatten()
    idx  = stratified_subsample(X, y, max_pts)
    X_s, y_s = X[idx], y[idx]
    n_pc  = min(50, X_s.shape[0]-1, X_s.shape[1])
    X_pca = PCA(n_components=n_pc, random_state=SEED).fit_transform(X_s)
    X_2d  = TSNE(n_components=2, perplexity=30, max_iter=1000,
                 random_state=SEED).fit_transform(X_pca)
    return X_2d, y_s

tsne_sel = widgets.Dropdown(
    options=STAGE_NAMES + ['Tous les stages'],
    description='Stage :', style={'description_width':'initial'},
    layout=widgets.Layout(width='260px'))
tsne_btn = widgets.Button(description='Lancer t-SNE',
                           button_style='warning', icon='play')
out_tsne = widgets.Output()

def on_tsne(b):
    if not STATE['features']:
        with out_tsne:
            print('Extraire les features d abord.')
        return
    sel    = tsne_sel.value
    stages = STAGE_NAMES if sel == 'Tous les stages' else [sel]
    with out_tsne:
        clear_output(wait=True)
        ncols = min(2, len(stages))
        nrows = (len(stages) + 1) // 2
        fig, axes = plt.subplots(nrows, ncols, figsize=(7*ncols, 6*nrows))
        axes_flat = np.array(axes).flatten()
        for i, sn in enumerate(stages):
            ax = axes_flat[i]
            print(f't-SNE {sn}...', end=' ', flush=True)
            t0 = time.time()
            X_2d, y_s = run_tsne(sn)
            print(f'{time.time()-t0:.1f}s')
            classes = np.unique(y_s)
            colors  = class_colors(classes)
            for cls in classes:
                mask = y_s == cls
                ax.scatter(X_2d[mask,0], X_2d[mask,1],
                           c=colors[int(cls)], s=6, alpha=0.7,
                           label=f'cls {int(cls)}')
            star = ' *' if sn == 'Stage 3' else ''
            ax.set_title(f'{sn}{star} — {len(classes)} classes — {len(y_s)} pts',
                          fontsize=11)
            ax.legend(markerscale=2, fontsize=8, loc='best')
            ax.set_xticks([]); ax.set_yticks([])
        for j in range(len(stages), nrows*ncols):
            axes_flat[j].axis('off')
        plt.suptitle('t-SNE — Embeddings colories par classe GT', fontsize=13)
        plt.tight_layout()
        plt.show()

tsne_btn.on_click(on_tsne)
display(widgets.HBox([tsne_sel, tsne_btn]), out_tsne)

Output()

---
## 6️⃣  Variance expliquée PCA (30 composantes)

Barres = variance par composante. Courbe rouge = cumul.  
La ligne orange marque PC3 (les 3 composantes utilisées pour la visualisation RGB).

In [ ]:
pca_var_btn = widgets.Button(description='Calculer variance PCA',
                              button_style='info', icon='bar-chart')
out_pca_var = widgets.Output()

def show_pca_variance(b):
    if not STATE['features']:
        with out_pca_var:
            print('Extraire les features d abord.')
        return
    with out_pca_var:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 4, figsize=(21, 4.5))
        for i, sn in enumerate(STAGE_NAMES):
            feat = STATE['features'][sn]
            X    = l2_norm(feat.reshape(-1, feat.shape[-1]))
            n_c  = min(30, X.shape[0]-1, X.shape[1])
            pca  = PCA(n_components=n_c, random_state=SEED).fit(X)
            evr  = pca.explained_variance_ratio_
            cum  = np.cumsum(evr)
            ax   = axes[i]
            ax.bar(range(1, n_c+1), evr*100, alpha=0.7,
                   color='steelblue', label='var/comp')
            ax2 = ax.twinx()
            ax2.plot(range(1, n_c+1), cum*100, 'r-', lw=2, label='cumul')
            ax2.set_ylim(0, 105)
            ax2.set_ylabel('Cumul (%)', color='red', fontsize=8)
            ax2.tick_params(axis='y', colors='red')
            ax.axvline(x=3, color='orange', ls='--', lw=1.5, alpha=0.8)
            sz   = STAGE_SIZES[sn]
            star = ' *' if sn == 'Stage 3' else ''
            ax.set_title(f'{sn}{star} ({sz}x{sz})\n'
                          f'PC1-3: {cum[2]*100:.1f}%  |  PC1-10: {cum[9]*100:.1f}%',
                          fontsize=9)
            ax.set_xlabel('Composante PCA', fontsize=8)
            ax.set_ylabel('Var. expliquee (%)', fontsize=8)
        plt.suptitle('Variance expliquee par composante PCA — 4 stages', fontsize=13)
        plt.tight_layout()
        plt.show()

pca_var_btn.on_click(show_pca_variance)
display(pca_var_btn, out_pca_var)

Button(button_style='info', description='Calculer variance PCA', icon='bar-chart', style=ButtonStyle())

Output()

---
## 7️⃣  Stage 3 ★ — Superposition PCA RGB + contours GT

Vérifie si les frontières de couleur dans la PCA RGB coïncident avec les zones GT.

In [ ]:
overlay_btn = widgets.Button(description='Afficher overlay Stage 3',
                              button_style='', icon='eye')
out_overlay = widgets.Output()

def show_overlay(b):
    if 'Stage 3' not in STATE['features']:
        with out_overlay:
            print('Extraire les features d abord.')
        return
    feat = STATE['features']['Stage 3']
    gt   = STATE['lbl_arr']
    rgb, _ = compute_pca_rgb(feat)
    orig_h, orig_w = gt.shape
    # Upscale PCA 64x64 -> 256x256
    pca_big = np.array(Image.fromarray(
        (rgb * 255).astype(np.uint8)
    ).resize((orig_w, orig_h), Image.BILINEAR)) / 255.0
    with out_overlay:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))
        axes[0].imshow(np.array(STATE['img_pil'].convert('RGB')))
        axes[0].set_title('Image originale', fontsize=11)
        axes[0].axis('off')
        axes[1].imshow(rgb)
        axes[1].set_title('PCA RGB Stage 3 (64x64)', fontsize=11, fontweight='bold')
        axes[1].axis('off')
        cmap_gt = plt.colormaps['tab10']
        classes = np.unique(gt)
        gt_rgb  = np.zeros((*gt.shape, 3))
        for k, cls in enumerate(classes):
            gt_rgb[gt == cls] = mcolors.to_rgb(cmap_gt(k))
        axes[2].imshow(gt_rgb)
        axes[2].set_title('GT mask', fontsize=11)
        axes[2].axis('off')
        axes[3].imshow(pca_big)
        try:
            from skimage.segmentation import find_boundaries
            for k, cls in enumerate(classes):
                mask = (gt == cls).astype(np.uint8)
                bd   = find_boundaries(mask, mode='outer')
                c    = mcolors.to_rgb(cmap_gt(k))
                ov   = np.zeros((*gt.shape, 4))
                ov[bd, :3] = c
                ov[bd, 3]  = 1.0
                axes[3].imshow(ov)
        except ImportError:
            for k, cls in enumerate(classes):
                axes[3].contour((gt == cls).astype(float),
                                levels=[0.5],
                                colors=[mcolors.to_hex(cmap_gt(k))],
                                linewidths=1.2)
        axes[3].set_title('PCA RGB Stage 3 + contours GT', fontsize=11)
        axes[3].axis('off')
        plt.suptitle('Stage 3 — Correspondance PCA RGB vs zones GT', fontsize=13)
        plt.tight_layout()
        plt.show()

overlay_btn.on_click(show_overlay)
display(overlay_btn, out_overlay)

Button(description='Afficher overlay Stage 3', icon='eye', style=ButtonStyle())

Output()

---
## 8️⃣  Comparaison : features pures vs features fusionnées FPN

**Architecture de la fusion (FPN top-down) :**
```
Stage 4 (32×32)  ──conv[0]──► lateral_4  ──────────────────────► out[3]  (pur)
                                  │
                              upsample ×2
                                  │
Stage 3 (64×64)  ──conv[1]──► lateral_3  ──+──► fused_3  ────────► out[2]  (fusionné ★)
Stage 2 (128×128)──conv[2]──► lateral_2  ──────────────────────► out[1]  (pur)
Stage 1 (256×256)──conv[3]──► lateral_1  ──────────────────────► out[0]  (pur)
```
`fpn_top_down_levels = [2, 3]` → **seul Stage 3 reçoit la fusion réelle**.  
Pour Stage 1 et Stage 2, *pur = fusionné* (tenseurs identiques).

In [11]:
# ── PCA RGB : pur vs fusionné ───────────────────────────────────────────
FUSED_NAMES = {'Stage 1': 'Stage 1 fus',
               'Stage 2': 'Stage 2 fus',
               'Stage 3': 'Stage 3 fus'}

pca_cmp_toggle = widgets.ToggleButtons(
    options=['Stage 1', 'Stage 2', 'Stage 3'],
    description='Stage :', button_style='',
    style={'button_width': '100px', 'description_width': 'initial'})
out_pca_cmp = widgets.Output()

def show_pca_cmp(change=None):
    if not STATE['features'] or not STATE['features_fused']:
        with out_pca_cmp:
            print('Extraire les features d abord.')
        return
    sn     = pca_cmp_toggle.value
    fn     = FUSED_NAMES[sn]
    sz     = STAGE_SIZES[sn]
    gt     = STATE['gt_maps'][sn]
    feat_p = STATE['features'][sn]
    feat_f = STATE['features_fused'][fn]
    rgb_p, evr_p = compute_pca_rgb(feat_p)
    rgb_f, evr_f = compute_pca_rgb(feat_f)
    # Verifier si les tenseurs sont identiques
    identical = np.allclose(feat_p, feat_f, atol=1e-5)
    with out_pca_cmp:
        clear_output(wait=True)
        if identical:
            print(f'INFO : {sn} pur == {sn} fus (pas de fusion top-down a ce niveau)')
        fig, axes = plt.subplots(1, 4, figsize=(22, 5))
        # Col 0 : image originale
        axes[0].imshow(np.array(STATE['img_pil'].convert('RGB')))
        axes[0].set_title('Image originale', fontsize=10)
        axes[0].axis('off')
        # Col 1 : PCA RGB pur
        axes[1].imshow(rgb_p)
        axes[1].set_title(
            f'{sn} pur ({sz}x{sz})\n'
            f'EV: {evr_p.sum()*100:.1f}%  '
            f'(PC1={evr_p[0]*100:.1f}% PC2={evr_p[1]*100:.1f}% PC3={evr_p[2]*100:.1f}%)',
            fontsize=9)
        axes[1].axis('off')
        # Col 2 : PCA RGB fusionné
        border_col = 'gold' if not identical else 'gray'
        for sp in axes[2].spines.values():
            sp.set_edgecolor(border_col); sp.set_linewidth(3)
        axes[2].imshow(rgb_f)
        fus_label = 'fusionné ★' if not identical else 'fusionné (= pur)'
        axes[2].set_title(
            f'{sn} {fus_label} ({sz}x{sz})\n'
            f'EV: {evr_f.sum()*100:.1f}%  '
            f'(PC1={evr_f[0]*100:.1f}% PC2={evr_f[1]*100:.1f}% PC3={evr_f[2]*100:.1f}%)',
            fontsize=9)
        axes[2].axis('off')
        # Col 3 : GT
        cmap_gt = plt.colormaps['tab10']
        classes = np.unique(gt)
        gt_rgb  = np.zeros((*gt.shape, 3))
        for k, cls in enumerate(classes):
            gt_rgb[gt == cls] = mcolors.to_rgb(cmap_gt(k))
        axes[3].imshow(gt_rgb)
        axes[3].set_title(f'GT @ {sz}x{sz}', fontsize=10)
        axes[3].axis('off')
        plt.suptitle(
            f'PCA RGB — {sn} pur vs fusionné' +
            ('' if not identical else '  [identiques : pas de fusion à ce niveau]'),
            fontsize=13)
        plt.tight_layout()
        plt.show()
        # Afficher distance cosine entre les deux
        if not identical:
            X_p = l2_norm(feat_p.reshape(-1, 256))
            X_f = l2_norm(feat_f.reshape(-1, 256))
            cos_sim = np.einsum('nd,nd->n', X_p, X_f).mean()
            print(f'Similarité cosine pixel-à-pixel (pur vs fus) : {cos_sim:.4f}')
            print(f'  1.0 = identiques, 0.0 = orthogonaux')

pca_cmp_toggle.observe(show_pca_cmp, names='value')
display(pca_cmp_toggle, out_pca_cmp)
show_pca_cmp()

ToggleButtons(description='Stage :', options=('Stage 1', 'Stage 2', 'Stage 3'), style=ToggleButtonsStyle(butto…

Output()

In [ ]:
# ── t-SNE comparatif : pur vs fusionné ──────────────────────────────────
tsne_cmp_toggle = widgets.ToggleButtons(
    options=['Stage 1', 'Stage 2', 'Stage 3'],
    description='Stage :', button_style='',
    style={'button_width': '100px', 'description_width': 'initial'})
tsne_cmp_btn = widgets.Button(description='Lancer t-SNE comparatif',
                               button_style='warning', icon='play')
out_tsne_cmp = widgets.Output()

def on_tsne_cmp(b):
    if not STATE['features'] or not STATE['features_fused']:
        with out_tsne_cmp:
            print('Extraire les features d abord.')
        return
    sn     = tsne_cmp_toggle.value
    fn     = FUSED_NAMES[sn]
    gt     = STATE['gt_maps'][sn]
    feat_p = STATE['features'][sn]
    feat_f = STATE['features_fused'][fn]
    identical = np.allclose(feat_p, feat_f, atol=1e-5)
    with out_tsne_cmp:
        clear_output(wait=True)
        if identical:
            print(f'INFO : {sn} pur == {sn} fus — t-SNE identique (pas de fusion à ce niveau)')
        fig, axes = plt.subplots(1, 2, figsize=(16, 7))
        for col, (feat, label) in enumerate([
                (feat_p, f'{sn} pur'),
                (feat_f, f'{sn} fusionné{" ★" if not identical else " (= pur)"}'),
        ]):
            ax = axes[col]
            X  = l2_norm(feat.reshape(-1, feat.shape[-1]))
            y  = gt.flatten()
            idx = stratified_subsample(X, y, 2000)
            X_s, y_s = X[idx], y[idx]
            print(f't-SNE {label} ({len(y_s)} pts)...', end=' ', flush=True)
            t0 = time.time()
            n_pc  = min(50, X_s.shape[0]-1, X_s.shape[1])
            X_pca = PCA(n_components=n_pc, random_state=SEED).fit_transform(X_s)
            X_2d  = TSNE(n_components=2, perplexity=30, max_iter=1000,
                         random_state=SEED).fit_transform(X_pca)
            print(f'{time.time()-t0:.1f}s')
            classes = np.unique(y_s)
            colors  = class_colors(classes)
            for cls in classes:
                mask = y_s == cls
                ax.scatter(X_2d[mask,0], X_2d[mask,1],
                           c=colors[int(cls)], s=6, alpha=0.7,
                           label=f'cls {int(cls)}')
            ax.set_title(f'{label}\n{len(classes)} classes — {len(y_s)} pts',
                          fontsize=11)
            ax.legend(markerscale=2, fontsize=8)
            ax.set_xticks([]); ax.set_yticks([])
            # Calcul séparabilité inter-classe
            centroids = {int(c): X_s[y_s==c].mean(axis=0) for c in classes}
            cls_list  = list(centroids.keys())
            dists = []
            for ii in range(len(cls_list)):
                for jj in range(ii+1, len(cls_list)):
                    ci = centroids[cls_list[ii]]; ci /= np.linalg.norm(ci)+1e-8
                    cj = centroids[cls_list[jj]]; cj /= np.linalg.norm(cj)+1e-8
                    dists.append(1 - np.dot(ci, cj))
            ax.set_xlabel(
                f'Dist. cosine inter-classe : moy={np.mean(dists):.4f}  '
                f'min={np.min(dists):.4f}  max={np.max(dists):.4f}',
                fontsize=8)
        plt.suptitle(
            f't-SNE comparatif — {sn} pur vs fusionné',
            fontsize=13)
        plt.tight_layout()
        plt.show()

tsne_cmp_btn.on_click(on_tsne_cmp)
display(widgets.HBox([tsne_cmp_toggle, tsne_cmp_btn]), out_tsne_cmp)

Output()